In [11]:
import pandas as pd 
from config import supp_table_path, llm_review_path
import matplotlib.pyplot as plt
import copy
import os 


In [12]:
# supp table alpha, likely table S2 

dms_performance_results = pd.read_csv(supp_table_path + "all_model_performance_DMS.csv")
dms_entropy_results = pd.read_csv(supp_table_path + "average_site_entropy_DMS.csv")
model_id_pair_to_entropy = {}
for index, row in dms_entropy_results.iterrows():
    identifier = row["Model"] + "_" + row["Identifier"]
    model_id_pair_to_entropy[identifier] = row["Average Site Entropy"]

dms_performance_results["joint_identifier"] = dms_performance_results["Model"] + "_" + dms_performance_results["Identifier"]
dms_performance_results["Average Site Entropy"] = dms_performance_results["joint_identifier"].apply(lambda x : model_id_pair_to_entropy[x])
pg_sequenecs = pd.read_csv(llm_review_path + "data/ProteinGym_reference_file_substitutions.csv")
pg_id_to_seq = dict(zip(pg_sequenecs["DMS_id"].values, pg_sequenecs["target_seq"].values))
dms_performance_results["Sequence"] = dms_performance_results["Identifier"].apply(lambda x : pg_id_to_seq[x])
dms_performance_results = dms_performance_results.drop(columns = ["joint_identifier"])



cat_site_performance_results = pd.read_csv(supp_table_path + "all_model_performance_cat_site.csv")
cat_site_entropy_results = pd.read_csv(supp_table_path + "average_site_entropy_cat_sites.csv")

model_id_pair_to_entropy = {}
for index, row in cat_site_entropy_results.iterrows():
    identifier = row["Model"] + "_" + row["Identifier"]
    model_id_pair_to_entropy[identifier] = row["Average Site Entropy"]

cat_site_performance_results["joint_identifier"] = cat_site_performance_results["Model"] + "_" + cat_site_performance_results["Identifier"]
cat_site_performance_results["Average Site Entropy"] = cat_site_performance_results["joint_identifier"].apply(lambda x : model_id_pair_to_entropy[x])
biolip_cleaned = pd.read_csv(llm_review_path + "data/biolip/biolip_cleaned.csv.gz")
biolip_cleaned["identifier"] = biolip_cleaned["PDB_ID"] + "_" + biolip_cleaned["Receptor_chain"]
biolip_id_to_seq = dict(zip(biolip_cleaned["identifier"].values, biolip_cleaned["Receptor_sequence"].values))
cat_site_performance_results["Sequence"] = cat_site_performance_results["Identifier"].apply(lambda x : biolip_id_to_seq[x])
cat_site_performance_results = cat_site_performance_results.drop(columns = ["joint_identifier"])



conservation_performance_results = pd.read_csv(supp_table_path + "all_model_performance_conservation.csv")
conservation_entropy_results = pd.read_csv(supp_table_path + "average_site_entropy_conservation.csv")
model_id_pair_to_entropy = {}
for index, row in conservation_entropy_results.iterrows():
    identifier = row["Model"] + "_" + row["Identifier"]
    model_id_pair_to_entropy[identifier] = row["Average Site Entropy"]
conservation_performance_results["joint_identifier"] = conservation_performance_results["Model"] + "_" + conservation_performance_results["Identifier"]
conservation_performance_results["Average Site Entropy"] = conservation_performance_results["joint_identifier"].apply(lambda x : model_id_pair_to_entropy[x])
conservation_sequences = pd.read_csv(llm_review_path + "data/Vogelstein2013_125drivers_sequences.csv", index_col = 0)
vogelstein_id_to_seq = dict(zip(conservation_sequences["gene"].values, conservation_sequences["sequence"].values))
conservation_performance_results["Sequence"] = conservation_performance_results["Identifier"].apply(lambda x : vogelstein_id_to_seq[x])
conservation_performance_results = conservation_performance_results.drop(columns = ["joint_identifier"])


combined = pd.concat([dms_performance_results, cat_site_performance_results, conservation_performance_results])
combined.to_csv(supp_table_path + "supp_table_alpha.csv")

In [13]:
# supp table beta, likely table S3
# performance relationships between all model pairs

df = pd.read_csv(supp_table_path + "model_realtionship_correlations_zero_shot.csv")
df = df.loc[df["Model 1"] != df["Model 2"]] # only removing these meaningless lines
df.to_csv(supp_table_path + "supp_table_beta.csv")



In [14]:
# supp table epsilon , skipped delta and gamma since used elsewhere 
# optimal model choices based on entropy for all tasks 

dms_maple_choices = pd.read_csv(supp_table_path + "entropy_optimal_model_choices_DMS.csv", index_col = 0)
cat_site_maple_choices = pd.read_csv(supp_table_path + "entropy_optimal_model_choices_cat_sites.csv")
conservation_model_choices = pd.read_csv(supp_table_path + "entropy_optimal_model_choices_conservation.csv", index_col = 0)

combined = pd.concat([dms_maple_choices, cat_site_maple_choices, conservation_model_choices])
combined.to_csv(supp_table_path + "supp_table_epsilon.csv")

In [15]:
# supp table zeta
# homologs by our definition for zero shot tasks 

lst = [
    "homologs_all_datasets_DMS.csv",
    "homologs_all_datasets_conservation.csv",
    "homologs_all_datasets_cat_sites.csv"
]

df_pieces = []
for fn in lst:
    df = pd.read_csv(supp_table_path + fn)
    df_pieces.append(df)
    
combined = pd.concat(df_pieces)
combined.to_csv(supp_table_path + "supp_table_zeta.csv")



In [16]:
# supp table eta
# alignment based scores, not valid for consetvation 


lst = [
    "aln_scores_all_datasets_DMS.csv",
    "aln_scores_all_datasets_cat_sites.csv",
]

df_pieces = []
for fn in lst:
    df = pd.read_csv(supp_table_path + fn)
    df_pieces.append(df)
    
combined = pd.concat(df_pieces)
combined.to_csv(supp_table_path + 'supp_table_eta.csv')

In [17]:
# supp table iota, skipped theta since used elsewhere 
# correlations between homologs and performance per  model per task

lst = [
    "homologs_v_performance_all_models_all_datasets_DMS.csv",
    "homologs_v_performance_all_models_all_datasets_conservation.csv",
    "homologs_v_performance_all_models_all_datasets_cat_sites.csv"
]


df_pieces = []
for fn in lst:
    df = pd.read_csv(supp_table_path + fn)
    df_pieces.append(df)
    
combined = pd.concat(df_pieces)

combined.to_csv(supp_table_path + 'supp_table_iota.csv')

In [18]:
# supp table kappa 
# maple based model selections for three zero shot tasks 
lst = [
    "entropy_optimal_model_choices_cat_sites.csv",
    "entropy_optimal_model_choices_conservation.csv",
    "entropy_optimal_model_choices_DMS.csv"
]

df_pieces = []
for fn in lst: 
    df = pd.read_csv(supp_table_path + fn)
    df_pieces.append(df)
combined = pd.concat(df_pieces)
combined = combined.drop(columns = ["Unnamed: 0"])
combined.to_csv(supp_table_path + 'supp_table_kappa.csv')

In [19]:
# genome wide selections using MAPLE 

g_w_maple = pd.read_csv(f"{llm_review_path}masked_results/entropy_optimal_files_human_proteins/metadata_1.0.csv.gz", index_col = 0)


shorthand_to_model_name = {
     'esm_2_15B' : 'ESM-2 (15B) UR50',
     'esm_2_3B': 'ESM-2 (3B) UR50',
     'esm_2_150M' : 'ESM-2 (150M) UR50',
     'esm_2_35M' : 'ESM-2 (35M) UR50',
     'esm_2_8M' : 'ESM-2 (8M) UR50',
     'esm_2_650M': 'ESM-2 (650M) UR50',
     'esm_1v': 'ESM-1v (650M) UR90',
     'protbert': 'ProtBERT (420M) UR100',
     'esm_1b': 'ESM-1b (650M) UR50',
     'esm_1_UR100': 'ESM-1 (670M) UR100',
     'esm_1_UR50D': 'ESM-1 (670M) UR50D',
     'esm_1_UR50S': 'ESM-1 (670M) UR50S',
     'esm_1_85M': 'ESM-1 (85M) UR50',
     'esm_1_43M': 'ESM-1 (43M) UR50',
}

g_w_maple["Model"] = g_w_maple["model_key"].apply(lambda x : shorthand_to_model_name[x])
g_w_maple = g_w_maple.rename(columns = {"selected_genes" : "Selected Genes", "num_selected_genes" : "Number of Selected Genes"})
g_w_maple = g_w_maple[["Model", "Selected Genes", "Number of Selected Genes"]]
g_w_maple.to_csv(supp_table_path + "genome_wide_maple_selections_gamma_1.0.csv", index = False)

In [20]:
writer = pd.ExcelWriter(llm_review_path + "figure_panels/submission/submission_tables.xlsx", engine = 'xlsxwriter')

# performance on zero shot tasks by model and seqeunece, with site entropy appended
alpha = pd.read_csv(supp_table_path  + "supp_table_alpha.csv", index_col = 0)
alpha.to_excel(writer, sheet_name = 'S1', index = False)


# performance relationships between all model pairs, on all zero shot tasks 
beta = pd.read_csv(supp_table_path  + "supp_table_beta.csv", index_col = 0)
beta.to_excel(writer, sheet_name = 'S2', index = False)


# homologs by our definition for zero shot tasks 
zeta = pd.read_csv(supp_table_path  + "supp_table_zeta.csv", index_col = 0)
zeta.to_excel(writer, sheet_name = 'S3', index = False)



# supp table iota, skipped theta since used elsewhere 
# correlations between homologs and performance per  model per task
iota = pd.read_csv(supp_table_path  + "supp_table_iota.csv", index_col = 0)
iota.to_excel(writer, sheet_name = 'S4', index = False)

# supp table eta
# alignment based scores, not valid for consetvation 
eta = pd.read_csv(supp_table_path  + "supp_table_eta.csv", index_col = 0)
eta.to_excel(writer, sheet_name = 'S5', index = False)


maple_train_experiments =  pd.read_csv(supp_table_path  + "entropy_optimal_training_experiments.csv")
maple_train_experiments = maple_train_experiments.drop(columns = list(filter(lambda x : "Unnamed" in x, list(maple_train_experiments.columns))))
maple_train_experiments.to_excel(writer, sheet_name = 'S6', index = False)


# supp table kappa
# MAPLE selections and performance for three zero shot tasks
kappa = pd.read_csv(supp_table_path  + "supp_table_kappa.csv", index_col = 0)
kappa.to_excel(writer, sheet_name = 'S7', index = False)


# MAPLE selections and performance for clinvar stuff
clinvar_maple = pd.read_csv(supp_table_path  + "clinvar_auroc_vals_with_MAPLE_annotations.csv", index_col = 0)
clinvar_maple.to_excel(writer, sheet_name = 'S8', index = False)


# MAPLE selections for genome wide scores
g_w_maple = pd.read_csv(supp_table_path +  "genome_wide_maple_selections_gamma_1.0.csv", index_col = 0)
g_w_maple.to_excel(writer, sheet_name = 'S9', index = False)


# long range contacts and entries, PDB files 
long_range_entries = pd.read_csv(supp_table_path + "long_range_entries.csv")
long_range_entries.to_excel(writer, sheet_name = "S10", index = False)


# conflicting ref aas, purple violin plots
conflicting_refs = pd.read_csv(supp_table_path + 'gene_ref_aas_different.csv')
conflicting_refs.to_excel(writer, sheet_name = 'S11', index = False)


# positional entropy tiled v. untiled 
pos_entropy_tiled_untiled = pd.read_csv(supp_table_path + "positional_entropies_1_through_1000_tiled_and_rotary.csv", index_col = 0)
pos_entropy_tiled_untiled.to_excel(writer, sheet_name = 'S12', index = False)

# clinvar long v short rotary 
clinvar_a = pd.read_csv(supp_table_path + "clinvar_misclass_summary_stats_with_scores_rotary_tiled.csv", index_col = 0)
# fixing typo
clinvar_a = clinvar_a.rename(columns  = {"variant_names_vus_long_rotary" : "variant_scores_vus_long_rotary"})
clinvar_a.to_excel(writer, sheet_name = 'S13', index = False)

# clinvar variants included in red, green, gray boxplots
clinvar_b = pd.read_csv(supp_table_path + "clinvar_misclassification_all_models_all_variants.csv", index_col = 0)
clinvar_b.to_excel(writer, sheet_name = 'S14', index = False)

# clinvar misclassification summary stats, tiled only, length independent choices of train/test 
clinvar_c = pd.read_csv(supp_table_path + "long_clinvar_classification_summary_stats.csv", index_col = 0)
clinvar_c.to_excel(writer, sheet_name = 'S15', index = False)


family_superfamily_fold_choices = pd.read_csv(supp_table_path + "scope_orphanage_choices_with_distances.csv", index_col = 0)
family_superfamily_fold_choices.to_excel(writer, sheet_name = 'S16', index = False)


# Sequences included from Orphan25, human selections, RandSeq, and Markov-based groups with varying window sizes
emb_sequences = pd.read_csv(supp_table_path + "embedding_space_used_seqeunces.csv")
# didnt end up using these
emb_sequences = emb_sequences.loc[~emb_sequences["group"].isin(["markov_based_sequences_window_13", "markov_based_sequences_window_14", "markov_based_sequences_window_15", "markov_based_sequences_window_16"])]
emb_sequences["is_orphanage_member"] = emb_sequences["group"].apply(lambda x : x != "Human Selections")
emb_sequences.to_excel(writer, sheet_name = 'S17', index = False)


# human vs. orphan25 seqs distances blue and green boxplots 
human_v_orphan25 = pd.read_csv(supp_table_path + "orphan_25_human_pair_distances.csv", index_col = 0)
human_v_orphan25.to_excel(writer, sheet_name = 'S18', index = False)


# Dimensional variance for embeddings of proteins from Orphan25, 25 human selections, bottom 25 proteins by homolog count in SCOPe, and top 25 proteins by homolog count in SCOPe
variance_dimensions = pd.read_csv(supp_table_path + "model_joined_protein_embedding_dimension_variance.csv", index_col = 0)
variance_dimensions.to_excel(writer, sheet_name = 'S19', index = False)


# SCOPe pairs comprising differnt folds 
diff_folds_SCOPE = pd.read_csv(supp_table_path + "top_bottom_homologs_pairs_scope_distances.csv")
diff_folds_SCOPE.to_excel(writer, sheet_name = 'S20', index = False)


intra_group_random_seqs = pd.read_csv(supp_table_path + "random_seq_intra_group_mean_pool_dists.csv")
# dont end up using these
intra_group_random_seqs = intra_group_random_seqs.loc[~intra_group_random_seqs["sequence_group"].isin(["markov_window_13", "markov_window_14", "markov_window_15", "markov_window_16"])]
intra_group_random_seqs.to_excel(writer, sheet_name = 'S21', index = False)


# info for 25000 random sequence selections 
random_ur50_25000 = pd.read_csv(supp_table_path + "orphanage_pcs_homologs_seqeunces_all_models_all_pooling.csv", index_col = 0)
random_ur50_25000.to_excel(writer, sheet_name = 'S22', index = False)


## human seqeunces used in all analyses 
human_sequences = pd.read_csv(llm_review_path + "data/uniprotkb_taxonomy_id_9606_AND_reviewed_2023_10_18.tsv", delimiter = "\t")
human_sequences.to_excel(writer, sheet_name = 'S23', index = False)


# supp table for biolip seqs and labels  
biolip_cleaned = pd.read_csv(llm_review_path + "data/biolip/biolip_cleaned.csv.gz", index_col = 0)
biolip_cleaned.to_excel(writer, sheet_name = 'S24', index = False)


writer.close()

# zip file, also keep uncompressed file 
if os.path.exists(llm_review_path + "figure_panels/submission/submission_tables.xlsx.gz"):
    os.system("rm " + llm_review_path + "figure_panels/submission/submission_tables.xlsx.gz" )

os.system("gzip -k -- " + llm_review_path + "figure_panels/submission/submission_tables.xlsx")

0